# Compare our fine-tuned checkpoint against the released public one  ·  Person B

Two separate comparisons, both read-only — nothing here trains or modifies either
checkpoint:

**A. HebNLI test-set accuracy.** `03_eval_nli.ipynb` measured `alephbert-hebnli-clean`
(ours) at accuracy 0.7961 / macro-F1 0.7940 on the 883-row clean test set. This
notebook measures the released checkpoint (`oriel9p/AlephBERT-FT-HebNLI-LCHAIM`,
what `nli_rerank.py` defaulted to before) on the *same* file, for a labelled
reference point.

**This is not a fair comparison, and the notebook says so at the point it matters:**
the released checkpoint was fine-tuned on *all* of HebNLI rather than a train/test
split, so this test set was very likely part of its own training data. A strong
score from it measures memorisation, not generalisation — `eval_nli.py` stamps a
`caveat` into its summary file for exactly this reason.

**B. The actual negation-probe experiment.** `nli_rerank.py` blends embedding cosine
with an NLI judgement; until now it always used the released checkpoint by default,
with no way to point the harness at ours. This notebook runs `src.harness.run_eval`
with `--interventions nli_rerank` once per checkpoint and puts both rows in the same
results table (`results/results_nli_rerank.csv`) — this is the one that answers
"does our decontaminated model actually help the probe measurement."

Runs in two places (Colab web UI, VS Code + Colab extension), same as the other
notebooks — the checkpoint lives on Drive and both embedder downloads and the
released checkpoint need a GPU-having session. Run cell by cell.

## Where everything ends up

| what | where | survives a reset? |
|---|---|---|
| cloned repo | VM disk | no |
| `data/raw/hebnli_test_clean.jsonl` | VM disk, regenerated from source | no |
| our checkpoint | **Drive**, read-only | yes |
| the released checkpoint | downloaded from the HF hub, cached on VM disk | no |
| `results/nli_test_public_baseline.json` + predictions csv | VM disk, then downloaded | via git |
| `results/results_nli_rerank.csv` | VM disk, then downloaded | via git |

Nothing here writes to `checkpoints/` on Drive.

## 1. Setup

**Prints** &nbsp; Nothing. This cell only defines a helper.

**Writes** &nbsp; Nothing.

In [1]:
# Secrets three ways: Colab's store, then the environment, then a prompt. The VS Code
# extension cannot reach Colab's secret store, so the fallbacks are what make this
# notebook portable. getpass also keeps the token out of the saved output.
import os, subprocess, getpass

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f'{name}: from Colab secrets')
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        print(f'{name}: from environment')
        return value.strip()
    return getpass.getpass(f'{name}: ').strip()

**Prints** &nbsp; Python and torch versions, the GPU name, and which `google.colab`
modules import. `cuda: True` plus a real GPU are worth confirming before the released
checkpoint's own download, which is a few hundred MB.

**Writes** &nbsp; Nothing.

In [2]:
# What are we running on? Answers 'will this work here' before anything slow.
import platform
print('python      ', platform.python_version())
print('cwd         ', os.getcwd())
try:
    import torch
    print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu         ', torch.cuda.get_device_name(0))
except ImportError:
    print('torch        not installed yet')
for mod in ('google.colab.userdata', 'google.colab.drive', 'google.colab.files'):
    try:
        __import__(mod)
        print(f'{mod:24s} available')
    except Exception as exc:
        print(f'{mod:24s} NOT available ({type(exc).__name__})')

python       3.12.13
cwd          /content
torch        2.11.0+cu128 | cuda: True
gpu          Tesla T4
google.colab.userdata    available
google.colab.drive       available
google.colab.files       available


**Prints** &nbsp; `GH_TOKEN:` and where it came from, pip's log, then the last 3
commits — the top one should be the newest `nli:` commit.

**Writes** &nbsp; The repo at `/content/hebrew-negation-embeddings` on the VM. The
working directory moves into it, so every path after this is relative to the repo root.

In [ ]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'main'

gh_token = get_secret('GH_TOKEN')
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'
bare_url = f'https://github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else '.')

# Re-authenticate origin before every fetch, not just on first clone: the repo is
# private, and the last step below strips the token, so a second run in the same
# session would otherwise fetch unauthenticated and fail.
subprocess.run(['git','remote','set-url','origin', url], check=True)
subprocess.run(['git','fetch','-q','origin',BRANCH], check=True)

# reset --hard, not pull: this VM's checkout is scratch space that always mirrors
# origin, never its own source of truth (every notebook here commits from your
# machine, not from the VM), so local drift must never block picking up a new
# push. `pull` (a merge) refuses whenever an untracked file would be overwritten,
# and scripts in this project routinely create exactly that - writing straight
# into results/ before those paths exist in git - so a later sync hits "untracked
# working tree files would be overwritten by merge" the moment that path is
# committed for real.
subprocess.run(['git','reset','-q','--hard', f'origin/{BRANCH}'], check=True)

# drop the token from the stored remote so it is not left on the VM's disk
subprocess.run(['git','remote','set-url','origin', bare_url], check=True)
del gh_token, url

!pip install -q -r requirements.txt
!git log --oneline -3

**Prints** &nbsp; `all pipeline checks passed`, `all NLI data checks passed`,
`all NLI eval checks passed`, `all run_eval checks passed`. Anything else: stop here.

**Writes** &nbsp; Nothing.

In [ ]:
# offline checks first - seconds, no network, no GPU
!python -m tests.test_data_pipeline | tail -3
!python -m tests.test_nli_data | tail -3
!python -m tests.test_nli_eval | tail -3
!python -m tests.test_run_eval | tail -3

## 2. Data — the clean HebNLI test split

Same file `03_eval_nli.ipynb` used: `data/raw/hebnli_test_clean.jsonl`, 883 rows,
built by dropping the 689 held-out promptIDs and auditing for text overlap with the
probe. Gitignored, so a fresh VM has to rebuild it — this checks first and only
redoes the work if it is missing or the wrong size.

**Prints** &nbsp; Whether the file is already here with the right count.

**Writes** &nbsp; Nothing.

In [ ]:
from pathlib import Path

TEST_CLEAN = Path('data/raw/hebnli_test_clean.jsonl')
n_existing = sum(1 for _ in TEST_CLEAN.open(encoding='utf-8')) if TEST_CLEAN.exists() else 0

NEEDS_REGEN = n_existing != 883
print(f'{TEST_CLEAN}: {n_existing} rows found' if n_existing else f'{TEST_CLEAN}: not found')
print('regeneration needed:', NEEDS_REGEN)

**Prints** &nbsp; `HF_TOKEN:` and where it came from — skipped if not needed.

**Writes** &nbsp; Nothing. The token stays in memory.

In [ ]:
if NEEDS_REGEN:
    os.environ['HF_TOKEN'] = get_secret('HF_TOKEN')
else:
    print('skipped - clean test file already present with the right row count')

**Prints** &nbsp; Skipped if not needed. Otherwise the same funnel as
`03_eval_nli.ipynb`: `rows 884` then `kept 883`.

**Writes** &nbsp; `data/raw/hebnli_test.jsonl` and `data/raw/hebnli_test_clean.jsonl` on
the VM, gitignored.<br>`results/nli_data_test.json` — the manifest, committed.

In [ ]:
if NEEDS_REGEN:
    !python -m src.data.hebnli --split test --out data/raw/hebnli_test.jsonl
    !python -m src.nli.prepare_data --source data/raw/hebnli_test.jsonl --split test --out data/raw/hebnli_test_clean.jsonl
else:
    print('skipped - nothing to regenerate')

**Prints** &nbsp; The funnel, then the row count. Must read exactly `883`.

**Writes** &nbsp; Nothing. It only reads the manifest and the file back.

In [ ]:
import json

manifest = json.load(open('results/nli_data_test.json', encoding='utf-8'))
f = manifest['funnel']
print(f"loaded={f['loaded']}  id_filter=-{f['loaded']-f['prompt_id_clean']}"
      f"  text_audit=-{manifest['text_overlap']['rows_dropped']}  kept={manifest['rows_written']}")

n_rows = sum(1 for _ in TEST_CLEAN.open(encoding='utf-8'))
assert n_rows == 883, f'expected exactly 883 rows in {TEST_CLEAN}, found {n_rows}'
assert manifest['rows_written'] == 883
print(f'\n{TEST_CLEAN}: {n_rows} rows confirmed')

## 3. Our checkpoint on Drive

**Prints** &nbsp; Drive's mount confirmation and a check that all four expected
files are there. No fallback: our checkpoint only exists on Drive.

**Writes** &nbsp; Nothing beyond mounting Drive at `/content/drive`.

In [ ]:
CKPT = '/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    raise RuntimeError(
        'Drive did not mount. Our checkpoint only exists on Drive - make sure this '
        f'session is signed in with the same Google account used for training. '
        f'({type(exc).__name__}: {exc})'
    ) from exc

ckpt_dir = Path(CKPT)
if not ckpt_dir.is_dir():
    raise RuntimeError(f'{CKPT} not found on Drive - check this mounted the right account')
for name in ('model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json'):
    if not (ckpt_dir / name).exists():
        raise RuntimeError(f'{name} missing from {CKPT}')
print('checkpoint dir:', CKPT)
print('found: model.safetensors, config.json, tokenizer.json, tokenizer_config.json')

## 4. Part A — HebNLI test-set accuracy: ours vs. the released checkpoint

Both against the same 883-row clean test file, so the numbers are directly
comparable in form — accuracy, macro-F1, per-class, confusion matrix. Whether the
*comparison* itself is fair is a separate question, addressed by the caveat printed
below the released checkpoint's numbers.

**Prints** &nbsp; Skipped with a note if `results/nli_test_alephbert-hebnli-clean.json`
already exists (from `03_eval_nli.ipynb`) — no need to burn GPU time twice for a
number that cannot change. Otherwise the same output `03_eval_nli.ipynb` produced.

**Writes** &nbsp; `results/nli_test_alephbert-hebnli-clean.json` and
`results/nli_test_predictions.csv`, only if not already present.

In [ ]:
OUR_SUMMARY = 'results/nli_test_alephbert-hebnli-clean.json'
if Path(OUR_SUMMARY).exists():
    print(f'skipped - {OUR_SUMMARY} already exists (from 03_eval_nli.ipynb)')
else:
    !python -m src.nli.eval_nli --checkpoint {CKPT} \
        --test data/raw/hebnli_test_clean.jsonl --expected-n 883 \
        --summary-out {OUR_SUMMARY} \
        --predictions-out results/nli_test_predictions.csv

**Prints** &nbsp; `label source assumed(released)`, `pair encoding joined`, then
accuracy / macro-F1 / per-class / confusion matrix, then the `[caveat]` line — read
that line before repeating this number anywhere.

**Writes** &nbsp; `results/nli_test_public_baseline.json` and
`results/nli_test_public_baseline_predictions.csv`, a few hundred KB, on the VM.

In [ ]:
!python -m src.nli.eval_nli --checkpoint oriel9p/AlephBERT-FT-HebNLI-LCHAIM \
    --test data/raw/hebnli_test_clean.jsonl --expected-n 883 \
    --summary-out results/nli_test_public_baseline.json \
    --predictions-out results/nli_test_public_baseline_predictions.csv

**Prints** &nbsp; Both summaries' headline numbers side by side.

**Writes** &nbsp; Nothing. It only reads the two summary files back.

In [ ]:
import json

ours = json.load(open('results/nli_test_alephbert-hebnli-clean.json', encoding='utf-8'))
public = json.load(open('results/nli_test_public_baseline.json', encoding='utf-8'))

print(f"{'metric':16s} {'ours (clean)':>14s} {'released':>14s}")
for key in ('accuracy', 'macro_precision', 'macro_recall', 'macro_f1'):
    print(f'{key:16s} {ours[key]:>14.4f} {public[key]:>14.4f}')

if public.get('caveat'):
    print(f"\n[caveat on 'released'] {public['caveat']}")

## 5. Part B — the negation-probe experiment: nli_rerank, both checkpoints

`nli_rerank.py` blends embedding cosine with `P(entailment) - P(contradiction)`:

    score = (1 - lam) * cosine + lam * nli_score

`lam` defaults to 1.0 (pure NLI), which means the embedder argument does not affect
`nli_rerank`'s score at all under the default — `cosine` is computed but never used
in the blend. So unlike the `baseline`/`projection` sweep, there is no reason to run
`nli_rerank` against all four frozen embedders; one is run below (`multilingual-e5`,
the strongest baseline performer) purely so the row has a `model` value, not because
the choice changes the number.

Two runs into the same `--out`: the released checkpoint (the current default), then
ours. `append_or_replace` (in `run_eval.py`) keys rows on
`(model, intervention, nli_checkpoint, nli_encoding, nli_lam)`, so the second run
adds a row beside the first instead of erasing it.

**Prints** &nbsp; The checkpoint path, `encoding joined` (or `pair`), the label
names, six pairs each `[ok]`/`[MISMATCH]`, and a tally — confirms the label mapping
before either harness run below trusts it.

**Writes** &nbsp; Nothing.

In [ ]:
print('=== released checkpoint ===')
!python -m src.interventions.check_nli_labels

print('\n=== our checkpoint ===')
!python -m src.interventions.check_nli_labels --model {CKPT} --subfolder ""

**Prints** &nbsp; `[ok] multilingual-e5 nli_rerank gap=... nevir=...`, then
`wrote 1 new/updated rows -> results/results_nli_rerank.csv (1 rows total)`.

**Writes** &nbsp; `results/results_nli_rerank.csv` — one row, the released checkpoint.

In [ ]:
!python -m src.harness.run_eval \
    --probe data/probe/probe.jsonl \
    --models multilingual-e5 --interventions nli_rerank \
    --out results/results_nli_rerank.csv

**Prints** &nbsp; The same shape, `(2 rows total)` this time.

**Writes** &nbsp; A second row in `results/results_nli_rerank.csv`, beside the first
rather than replacing it.

In [ ]:
!python -m src.harness.run_eval \
    --probe data/probe/probe.jsonl \
    --models multilingual-e5 --interventions nli_rerank \
    --nli-model {CKPT} --nli-subfolder "" --nli-encoding pair \
    --out results/results_nli_rerank.csv

**Prints** &nbsp; The two-row comparison table: `nli_checkpoint`,
`cosine_gap`, `nevir_rank` side by side for the released checkpoint and ours.

**Writes** &nbsp; Nothing. It only reads the csv back.

In [ ]:
import pandas as pd

df = pd.read_csv('results/results_nli_rerank.csv')
df[['nli_checkpoint', 'nli_encoding', 'cosine_gap', 'nevir_rank']]

## 6. Download the results

**Prints** &nbsp; Browser downloads, or the files printed inline when that is
unavailable — printed unconditionally this time (see `03_eval_nli.ipynb`'s section 7:
under the VS Code extension `files.download()` can return without raising while the
file lands nowhere findable, so the inline print is not gated on an exception).

**Writes** &nbsp; Your machine, as downloads and/or as text in the saved notebook.

In [ ]:
RESULTS = [
    'results/nli_test_public_baseline.json',
    'results/nli_test_public_baseline_predictions.csv',
    'results/results_nli_rerank.csv',
]
try:
    from google.colab import files
    for path in RESULTS:
        files.download(path)
except Exception as exc:
    print(f'[warn] browser download unavailable ({type(exc).__name__})')

for path in RESULTS:
    print(f'\n===== {path} =====')
    print(open(path, encoding='utf-8').read())

Commit the three result files from your machine with the `nli:` prefix.

`results/nli_test_alephbert-hebnli-clean.json` / `results/nli_test_predictions.csv`
are unchanged from `03_eval_nli.ipynb` if this run skipped re-computing them — nothing
new to commit there unless section 4's first cell actually ran.